In [1]:
import os
import s3fs
import pandas as pd
import geopandas as gpd

os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"
fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]},
    skip_instance_cache=True,
)

dossier = "acharki-ensae/Projet-analyse-BPE/processed"
with fs.open(f"{dossier}/bpe_alim_2025.parquet", "rb") as f:
    bpe = gpd.read_parquet(f)
with fs.open(f"{dossier}/dvf_ventes_2025.parquet", "rb") as f:
    dvf = gpd.read_parquet(f)

print("BPE :", bpe.shape, bpe.crs)
print("DVF :", dvf.shape, dvf.crs)

PermissionError: Forbidden

In [ ]:
!pip install scipy

In [ ]:
codes_b2 = sorted(c for c in bpe["TYPEQU"].unique() if c.startswith("B2"))

types_commerces = {
    "hyper": ["B104"],
    "super": ["B105"],
    "proximite": ["B201", "B202"],
    "specialise": [c for c in codes_b2 if c not in ["B201", "B202"]],
}

for nom, codes in types_commerces.items():
    print(nom, ":", codes, "->", bpe["TYPEQU"].isin(codes).sum(), "commerces")

In [ ]:
for nom, codes in types_commerces.items():
    commerces = bpe.loc[bpe["TYPEQU"].isin(codes), ["geometry"]]

    plus_proche_type = gpd.sjoin_nearest(
        dvf[["geometry"]], commerces,
        how="left", distance_col=f"dist_{nom}",
    )
    plus_proche_type = plus_proche_type[~plus_proche_type.index.duplicated()]

    dvf[f"dist_{nom}"] = plus_proche_type[f"dist_{nom}"]
    print(f"{nom} : distance médiane {dvf[f'dist_{nom}'].median():.0f} m")

In [ ]:
commerces_alim = bpe.loc[
    bpe["TYPEQU"].isin(sum(types_commerces.values(), [])),
    ["TYPEQU", "geometry"],
]

plus_proche = gpd.sjoin_nearest(
    dvf[["geometry"]], commerces_alim,
    how="left", distance_col="dist_plus_proche",
)
plus_proche = plus_proche[~plus_proche.index.duplicated()]

noms_types = {code: nom for nom, codes in types_commerces.items() for code in codes}
dvf["dist_plus_proche"] = plus_proche["dist_plus_proche"]
dvf["type_plus_proche"] = plus_proche["TYPEQU"].map(noms_types)

dvf["type_plus_proche"].value_counts(normalize=True).round(3)

In [ ]:
colonnes_dist = ["dist_hyper", "dist_super", "dist_proximite", "dist_plus_proche"]
dvf[colonnes_dist].describe(percentiles=[0.1, 0.5, 0.9]).round(0)

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

poids = {"B104": 3, "B105": 2}
bpe["poids"] = bpe["TYPEQU"].map(poids).fillna(1).astype(int)

bpe.groupby("TYPEQU")["poids"].first()

In [ ]:
densite7_communes = bpe.groupby("DEPCOM")["DENS7"].first()
dvf["DENS7"] = dvf["code_commune"].map(densite7_communes)

print(dvf["DENS7"].value_counts(dropna=False).sort_index())

In [ ]:
dvf["densite_imputee"] = dvf["DENS7"].isna()
print("Ventes sans densité connue :", dvf["densite_imputee"].sum())

dvf["DENS7"] = dvf["DENS7"].fillna("6")
dvf["DENS7"].value_counts().sort_index()

In [ ]:
PORTEES = {
    "1": 1_000, "2": 1_500, "3": 2_500, "4": 3_000,
    "5": 4_000, "6": 6_000, "7": 8_000,
}   # en mètres

dvf["indice_acces"] = np.nan
for niveau, portee in PORTEES.items():
    masque = (dvf["DENS7"] == niveau).to_numpy()
    if masque.sum() == 0:
        continue
    xy = np.column_stack([dvf.geometry.x[masque], dvf.geometry.y[masque]])
    dvf.loc[masque, "indice_acces"] = indice_acces(xy, portee, w_commerces)
    print(f"Densité {niveau} (portée {portee} m) : {masque.sum()} ventes traitées")

In [ ]:
print("Valeurs manquantes :", dvf["indice_acces"].isna().sum())
print("Ventes sans aucun commerce dans le rayon :", (dvf["indice_acces"] == 0).sum())

dvf.groupby("DENS7")["indice_acces"].describe(percentiles=[0.1, 0.5, 0.9]).round(2)


In [ ]:
dvf.nlargest(10, "indice_acces")[["nom_commune", "DENS7", "indice_acces", "prix_m2"]]

In [ ]:
dvf[dvf["nom_commune"].isin(["Piscop", "Bouqueval", "Roissy-en-France"])].groupby(
    ["nom_commune", "DENS7"]
)["indice_acces"].median().round(1)

In [ ]:
(dvf[dvf["DENS7"] == "6"]
 .groupby("densite_imputee")["indice_acces"]
 .describe(percentiles=[0.5, 0.9, 0.99])
 .round(1))

In [ ]:
lambda_commerces = bpe["DENS7"].map(PORTEES).to_numpy(dtype=float)
print("Commerces sans portée :", np.isnan(lambda_commerces).sum())

In [ ]:
def indice_acces_composite(xy_ventes, portee_vente, w, lambda_com):
    """Indice gravitaire avec la portée la plus contraignante entre logement et commerce."""
    resultat = np.zeros(len(xy_ventes))
    for debut in range(0, len(xy_ventes), TAILLE_BLOC):
        bloc = xy_ventes[debut:debut + TAILLE_BLOC]
        paires = cKDTree(bloc).sparse_distance_matrix(
            arbre_commerces, max_distance=4 * portee_vente, output_type="ndarray"
        )
        portee_paire = np.minimum(portee_vente, lambda_com[paires["j"]])
        contributions = w[paires["j"]] * np.exp(-paires["v"] / portee_paire)
        resultat[debut:debut + len(bloc)] = np.bincount(
            paires["i"], weights=contributions, minlength=len(bloc)
        )
    return resultat

In [ ]:
dvf["indice_acces"] = np.nan
for niveau, portee in PORTEES.items():
    masque = (dvf["DENS7"] == niveau).to_numpy()
    if masque.sum() == 0:
        continue
    xy = np.column_stack([dvf.geometry.x[masque], dvf.geometry.y[masque]])
    dvf.loc[masque, "indice_acces"] = indice_acces_composite(
        xy, portee, w_commerces, lambda_commerces
    )
    print(f"Densité {niveau} (portée {portee} m) : {masque.sum()} ventes traitées")

In [ ]:
dvf.groupby("DENS7")["indice_acces"].describe(percentiles=[0.1, 0.5, 0.9]).round(2)


In [ ]:
dvf.nlargest(10, "indice_acces")[["nom_commune", "DENS7", "indice_acces", "prix_m2"]]

In [ ]:
(dvf[dvf["nom_commune"].isin(["Piscop", "Bouqueval", "Toussus-le-Noble", "Roissy-en-France"])]
 .groupby(["nom_commune", "DENS7"])["indice_acces"].median().round(1))

In [ ]:
dvf["indice_rang"] = dvf.groupby("DENS7")["indice_acces"].rank(pct=True)

In [ ]:
import os
import subprocess

DOSSIER_LOCAL = "/home/onyxia/work/donnees"            # hors du dépôt Git
DOSSIER_BUCKET = "s3/acharki-ensae/Projet-analyse-BPE/processed"

def sauvegarder(df, nom_fichier, delai_max=300):
    """Sauvegarde df en Parquet en local, puis tente de l'envoyer dans le bucket."""
    # 1. Sauvegarde locale
    os.makedirs(DOSSIER_LOCAL, exist_ok=True)
    chemin_local = os.path.join(DOSSIER_LOCAL, nom_fichier)
    df.to_parquet(chemin_local)
    taille = os.path.getsize(chemin_local) / 1e6
    print(f"Sauvegarde locale OK : {chemin_local} ({taille:.1f} Mo)")

    # 2. Envoi vers le bucket, avec une limite de temps
    try:
        r = subprocess.run(
            ["mc", "cp", chemin_local, f"{DOSSIER_BUCKET}/{nom_fichier}"],
            capture_output=True, text=True, timeout=delai_max,
        )
        if r.returncode == 0:
            print(f"Envoi vers le bucket OK : {DOSSIER_BUCKET}/{nom_fichier}")
        else:
            print("Échec de l'envoi vers le bucket :", r.stderr.strip())
            
    except subprocess.TimeoutExpired:
        print(f"Le bucket n'a pas répondu en {delai_max} s : seule la copie locale existe.")

In [ ]:
sauvegarder(dvf, "dvf_enrichi_2025.parquet")

In [2]:
import os
import subprocess
from getpass import getpass

depot = "/home/onyxia/work/projet__python_DS"

def git(*args):
    r = subprocess.run(["git", "-C", depot, *args], capture_output=True, text=True)
    return r.stdout + r.stderr

print(git("pull"))
print(git("add", "notebooks/03_croisement.ipynb", "src/stockage.py", "requirements.txt"))
print(git("status", "--short"))
print(git("commit", "-m", "Travail en cours : nouvelle version de l'indice d'accessibilité"))

token = os.environ.get("GIT_PERSONAL_ACCESS_TOKEN") or getpass("Colle ton token GitHub : ")
url = f"https://{token}@github.com/Samsamdu86/projet__python_DS.git"
print(git("push", url, "main").replace(token, "***"))

Updating a3c84d1..fe14f31
From https://github.com/Samsamdu86/projet__python_DS
   a3c84d1..fe14f31  main       -> origin/main
error: Your local changes to the following files would be overwritten by merge:
	notebooks/01_nettoyage_bpe.ipynb
	notebooks/02_nettoyage_dvf.ipynb
Please commit your changes or stash them before you merge.
Aborting

fatal: pathspec 'src/stockage.py' did not match any files

 M notebooks/01_nettoyage_bpe.ipynb
 M notebooks/02_nettoyage_dvf.ipynb
 M requirements.txt
?? notebooks/03_croisement.ipynb

On branch main
Your branch is behind 'origin/main' by 9 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/01_nettoyage_bpe.ipynb
	modified:   notebooks/02_nettoyage_dvf.ipynb
	modified:   requirements.txt

Untracked files:
  (use "git add <file>..." t

Colle ton token GitHub :  ········


To https://github.com/Samsamdu86/projet__python_DS.git
 ! [rejected]        main -> main (non-fast-forward)
error: failed to push some refs to 'https://github.com/Samsamdu86/projet__python_DS.git'
hint: Updates were rejected because the tip of your current branch is behind
hint: its remote counterpart. If you want to integrate the remote changes,
hint: use 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.

